In [2]:
import os
import re
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.preprocessing import normalize

from umap import UMAP
import matplotlib.pyplot as plt

c:\Users\flori\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
TEXT_DIR = "archelec_ocr_texts"

def load_texts(df, text_dir):
    texts = []
    meta = []

    for _, row in df.iterrows():
        doc_id = str(row["id"])
        path = os.path.join(text_dir, f"{doc_id}.txt")

        if not os.path.exists(path):
            continue

        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        texts.append(text)
        meta.append({
            "id": doc_id,
            "candidate": f"{row.get('titulaire-prenom', '')} {row.get('titulaire-nom', '')}",
            "party": row.get("titulaire-liste"),
            "date": row.get("date")
        })

    return texts, pd.DataFrame(meta)


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

texts_clean = [clean_text(t) for t in texts]


In [ ]:
vectorizer = CountVectorizer(
    stop_words="french",
    min_df=10,
    max_df=0.7,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(texts_clean)

N_TOPICS = 15

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    learning_method="batch"
)

doc_topics = lda.fit_transform(X)


In [ ]:
def print_topics(model, vectorizer, n_words=10):
    words = vectorizer.get_feature_names_out()
    for i, topic in enumerate(model.components_):
        top_words = [words[j] for j in topic.argsort()[-n_words:][::-1]]
        print(f"Topic {i}: {', '.join(top_words)}")

print_topics(lda, vectorizer)


In [ ]:
doc_topics_norm = normalize(doc_topics, norm="l1")

umap = UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="hellinger",
    random_state=42
)

coords = umap.fit_transform(doc_topics_norm)

meta_df["x"] = coords[:, 0]
meta_df["y"] = coords[:, 1]


In [ ]:
plt.figure(figsize=(10, 8))

for party, g in meta_df.groupby("party"):
    if pd.isna(party):
        continue
    plt.scatter(g["x"], g["y"], label=party, alpha=0.6, s=15)

plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.title("Semantic map of electoral manifestos (Topic Modeling)")
plt.xlabel("Semantic axis 1")
plt.ylabel("Semantic axis 2")
plt.tight_layout()
plt.show()
